# Day 5 — Practice

---

Three exercises. Needs `TOGETHER_API_KEY` and `OPENAI_API_KEY` in `.env`.

In [ ]:
!pip install openai together python-dotenv --quiet

In [ ]:
import os, json
from dotenv import load_dotenv
from openai import OpenAI
from together import Together

load_dotenv()
tg = Together()
oa = OpenAI() if os.getenv("OPENAI_API_KEY") else None
TG_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo"
OA_MODEL = "gpt-4o-mini"

## Exercise 1 — Extract structured data from a review

Use Together AI JSON mode to extract from this review:

> *"I bought the NoiseCancel Pro headphones last week. The battery is amazing, easily 30 hours. But the app crashes constantly and the ear cups feel small. 3 out of 5."*

Into:
- `rating` (int, 1–5)
- `product` (string)
- `topics` (list of strings — the things reviewed)

In [ ]:
review = ("I bought the NoiseCancel Pro headphones last week. The battery is amazing, "
          "easily 30 hours. But the app crashes constantly and the ear cups feel small. "
          "3 out of 5.")

prompt = f'''Extract as JSON with keys: rating (int 1-5), product (string), topics (list of strings).
Reply with ONLY the JSON object.

Review: {review}'''

# TODO: call tg.chat.completions.create with response_format={"type":"json_object"}
# TODO: json.loads the reply and print each field

## Exercise 2 — Wire up a `calculate_tip` tool

Define a Python function `calculate_tip(bill, tip_percent)` that returns `{"tip": ..., "total": ...}`. Give the model the tool schema and ask:

> *"The dinner bill was $84 and I want to tip 18%. How much should I leave?"*

Print:
- The arguments the model chose
- The result of running the function
- The final answer the model wrote

In [ ]:
def calculate_tip(bill: float, tip_percent: float) -> dict:
    tip = round(bill * tip_percent / 100, 2)
    return {"tip": tip, "total": round(bill + tip, 2)}

TOOLS = [{
    "type": "function",
    "function": {
        "name": "calculate_tip",
        "description": "Compute tip amount and total for a restaurant bill.",
        "parameters": {
            "type": "object",
            "properties": {
                "bill": {"type": "number", "description": "Bill amount in dollars"},
                "tip_percent": {"type": "number", "description": "Tip percentage 0-100"},
            },
            "required": ["bill", "tip_percent"],
        },
    },
}]

# TODO: send the question with tools=TOOLS to oa
# TODO: extract tool call, run calculate_tip(**args), print result
# TODO: send result back with role='tool', print final answer

## Exercise 3 — Two tools, model picks the right one

Define both:
- `get_current_stock_price(symbol: str)`
- `send_email(to: str, subject: str, body: str)`

Ask two questions and print which tool the model chooses:

1. *"What's the current price of AAPL?"* — expect `get_current_stock_price`
2. *"Email alice@example.com saying I'm running late."* — expect `send_email`

You don't need to actually run the tools — just print `.tool_calls[0].function.name` for each.

In [ ]:
TOOLS3 = [
    {"type":"function", "function":{
        "name": "get_current_stock_price",
        "description": "Look up the current price of a stock.",
        "parameters": {"type":"object", "properties":{"symbol":{"type":"string"}}, "required":["symbol"]},
    }},
    {"type":"function", "function":{
        "name": "send_email",
        "description": "Send an email to a person.",
        "parameters": {"type":"object", "properties":{
            "to":{"type":"string"}, "subject":{"type":"string"}, "body":{"type":"string"},
        }, "required":["to","subject","body"]},
    }},
]

QUESTIONS = [
    "What's the current price of AAPL?",
    "Email alice@example.com saying I'm running late.",
]

# TODO: for each question, call oa.chat.completions.create with tools=TOOLS3
# TODO: print the tool name chosen and the args

## Optional stretch — validate and retry

Sometimes JSON mode still returns something you don't expect (e.g. wrong types). Try wrapping a JSON-extraction call in a `try/except` that catches `json.JSONDecodeError` and retries once with the instruction *"Your last reply was not valid JSON. Try again."*

---

✅ **Done.** Tool calling is the skill that turns AI from a chatbot into a real workflow engine. Tomorrow: keep costs low and replies fast.